# 01 数组与向量运算

**前置：**第00课。**目标：**创建并观察多维数组，理解形状、索引、切片、轴、广播与矩阵运算。**预计时间：**45—60分钟。**产物：**一份房屋特征预测小实验。

## 带着问题开始
如果每套房有面积和房龄两个特征，怎样一次算出多套房的估价？本课用合成数据练习，面积单位与价格均为教学用模拟单位，不对应真实房屋。

## 1. 从列表到数组
Python列表适合收纳不同类型对象；NumPy 数组（ndarray，N-dimensional array，多维数组）适合规则数值计算。数组元素通常共享数据类型（dtype，data type），维数由 ndim 表示，各维长度由 shape 表示。NumPy（Numerical Python，Python数值计算库）提供下列接口。

In [ ]:
import numpy as np

列表 = [2, 4, 6]
数组 = np.array(列表)  # array：创建数组
print(列表 * 2)
print(数组 * 2)
print(数组.dtype, 数组.ndim, 数组.shape)

[2, 4, 6, 2, 4, 6]
[ 4  8 12]
int64 1 (3,)


列表乘2会重复内容，数组乘2会逐元素乘法。访问元素用索引，切片的右端不包含在结果中。

In [ ]:
A = np.array([[10, 20], [30, 40], [50, 60]])
print(A[1, 0])
print(A[:2, 1])

30
[20 40]


## 2. 形状、向量与轴
本课用一维数组表示向量，用二维数组表示矩阵。轴（axis）指定沿哪条轴操作：对形状(3,2)的矩阵，axis=0沿行方向汇总、每列得到一个结果；axis=1沿列方向汇总、每行得到一个结果。

In [ ]:
print(A.shape)
print(A.sum(axis=0))  # 每列
print(A.sum(axis=1))  # 每行

(3, 2)
[ 90 120]
[ 30  70 110]


## 3. 逐元素运算与矩阵乘法
星号 `*` 对应逐元素乘法；`@` 对应矩阵乘法。两个二维矩阵相乘时，左矩阵的列数等于右矩阵的行数。矩阵乘一维向量时，矩阵列数等于向量长度，结果为每一行与向量的点积（dot product）。

In [ ]:
P = np.array([[1, 2], [3, 4]])
Q = np.array([[10, 20], [30, 40]])
print(P * Q)
print(P @ Q)

[[ 10  40]
 [ 90 160]]
[[ 70 100]
 [150 220]]


## 4. 广播：形状如何对齐
广播让较小数组按兼容维度参与计算。从最右侧维度比较，长度相同或其中一个为1即可。示例：

|形状|示意|
|---|---|
|(3,2)|a b / c d / e f|
|(2,)|x y|
|结果(3,2)|a+x b+y / c+x d+y / e+x f+y|

`(3,1)+(2,)`会把`(2,)`看作`(1,2)`，结果为`(3,2)`。

In [ ]:
左 = np.array([[1, 10], [2, 20], [3, 30]])
右 = np.array([100, 1000])
print(左 + 右)
列 = np.array([[10], [20], [30]])
print((列 + 右).shape)

[[ 101 1010]
 [ 102 1020]
 [ 103 1030]]
(3, 2)


## 5. 三种形状与差值陷阱
长度为2的向量形状是`(2,)`；单行矩阵形状是`(1,2)`；单列矩阵形状是`(2,1)`。`(n,) - (n,1)`会广播成`(n,n)`，产生两两差值表。若要逐项相减，先统一形状，例如都用`(n,)`。

### 把三种形状实际摆出来

一维数组没有单独的行轴或列轴。`reshape` 用于改变数组形状，`[:, None]` 增加一条轴。改变形状会影响与其他数组运算时的广播方式；先检查形状，再判断运算结果。

In [ ]:
v = np.array([2, 4])
for name, value in [("一维向量", v), ("一行矩阵", v.reshape(1, 2)), ("一列矩阵", v[:, None])]:
    print(name, "形状", value.shape)
    print(value)
assert v.shape == (2,)
assert v.reshape(1, 2).shape == (1, 2)
assert v[:, None].shape == (2, 1)

一维向量 形状 (2,)
[2 4]
一行矩阵 形状 (1, 2)
[[2 4]]
一列矩阵 形状 (2, 1)
[[2]
 [4]]


In [ ]:
预测 = np.array([3.0, 5.0, 8.0])
答案 = np.array([[2.0], [4.0], [7.0]])
print((预测 - 答案).shape)  # (3,3)，常见意外
答案一维 = 答案.ravel()
print(预测 - 答案一维)

(3, 3)
[1. 1. 1.]


## 6. 房屋特征预测
每行是一套房，每列依次为面积、房龄。权重由我们手动设定；本课没有训练模型，学习模型留到后续课程。

记特征矩阵为 $X$，权重向量为 $w$，偏置为 $b$，则预测为 $Xw+b$。每一行的面积乘面积权重，房龄乘房龄权重，再加偏置。面积权重为正、房龄权重为负，分别表示面积增加提高预测、房龄增加降低预测；这些方向来自本课手工设定。

In [ ]:
X = np.array([[80, 10], [120, 5], [60, 20]], dtype=float)
w = np.array([2.0, -0.5])
b = 10.0
预测价格 = X @ w + b
预测价格

array([165. , 247.5, 120. ])

In [ ]:
逐套计算 = []
for 房屋 in X:
    逐套计算.append(房屋[0] * w[0] + 房屋[1] * w[1] + b)
逐套计算 = np.array(逐套计算)
assert np.allclose(逐套计算, 预测价格)
print(逐套计算)

[165.  247.5 120. ]


## 主动实验一：改权重
把面积权重改为3，观察每套预测变化。先猜哪套变化最大，再运行。

In [ ]:
新权重 = np.array([3.0, -0.5])
新预测 = X @ 新权重 + b
print(新预测 - 预测价格)

[ 80. 120.  60.]


## 主动实验二：换偏置
把偏置改为20。观察每套预测的变化量是否相同，并解释原因。

In [ ]:
新偏置 = 20.0
新偏置预测 = X @ w + 新偏置
print(新偏置预测 - 预测价格)

[10. 10. 10.]


## 练习与自检
练习1：对矩阵`A`按列求均值，将答案写入唯一变量`exercise_column_means=None`后运行自检。参考解折叠在下方。

In [ ]:
exercise_column_means = None
if exercise_column_means is not None:
    assert np.allclose(exercise_column_means, np.array([30, 40]))

<details><summary>练习1参考解</summary>

```python
exercise_column_means = A.mean(axis=0)
assert np.allclose(exercise_column_means, [30, 40])
```
</details>

练习2：将面积权重改为1.5，计算预测并写入唯一变量`exercise_predictions=None`。

In [ ]:
exercise_predictions = None
if exercise_predictions is not None:
    assert np.allclose(exercise_predictions, X @ np.array([1.5, -0.5]) + b)

<details><summary>练习2参考解</summary>

```python
exercise_predictions = X @ np.array([1.5, -0.5]) + b
assert np.allclose(exercise_predictions, [125, 187.5, 90])
```
</details>

## 总结与迁移
掌握清单：能读出`dtype/ndim/shape`；能解释切片与axis；能区分`*`和`@`；能判断广播结果形状；能用手算和向量化互相核对。

上一课：[00 进入笔记本](00_进入笔记本.ipynb)；下一课：[02 数据与可视化](02_数据与可视化.ipynb)。

延伸阅读：[NumPy 入门](https://numpy.org/doc/stable/user/absolute_beginners.html)、[广播规则](https://numpy.org/doc/stable/user/basics.broadcasting.html)。